In [1]:
import pandas as pd

# Đọc dữ liệu từ file parquet
train_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\train_df_prepared.parquet", engine="pyarrow")
valid_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\valid_df_prepared.parquet", engine="pyarrow")
test_df = pd.read_parquet(r"C:\Users\Admin\Downloads\IoT Dataset\final_data_http\chunk-based-split-3\test_df_prepared.parquet", engine="pyarrow")

# Chuyển đổi timestamp sang số (giây)
train_df["timestamp_num"] = pd.to_datetime(train_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
valid_df["timestamp_num"] = pd.to_datetime(valid_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9
test_df["timestamp_num"] = pd.to_datetime(test_df['timestamp'], format='mixed', utc=True).astype('int64') / 1e9

# Sắp xếp tĩnh trực tiếp theo thời gian
train_df.sort_values(by='timestamp_num', inplace=True)
valid_df.sort_values(by='timestamp_num', inplace=True)
test_df.sort_values(by='timestamp_num', inplace=True)

print("Dữ liệu đã được load và sắp xếp theo thời gian thành công!")


Dữ liệu đã được load và sắp xếp theo thời gian thành công!


In [10]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import torch.optim as optim
from sklearn.metrics import accuracy_score, f1_score
from tqdm.notebook import tqdm
import gc

# =================================================================================
# 2. CẤU TRÚC DỮ LIỆU ĐỒ THỊ ĐỘNG (EDGE CLASSIFICATION)
# =================================================================================
def build_dynamic_snapshots(df, alpha=0.5, beta=0.5, delta_t=1.0, n_inactive=5, size_col='tot_len', label_col='label'):
    """
    Thuật toán giả lập đồ thị. Node = IP, Edge = Flow.
    Gắn nhãn (Label) cho Cạnh thay vì Node để thực hiện Flow Classification.
    """
    print(f"Khởi tạo Dynamic Snapshots (Edge Labels) với Δt={delta_t}s...")
    t_min = df['timestamp_num'].min()
    df['time_bin'] = ((df['timestamp_num'] - t_min) // delta_t).astype(int)
    
    all_ips = pd.concat([df['src_ip'], df['dst_ip']]).unique()
    ip_to_id = {ip: idx for idx, ip in enumerate(all_ips)}
    
    snapshots = []
    active_edges = {} 
    
    grouped = df.groupby('time_bin')
    max_bin = df['time_bin'].max()
    
    num_node_features = 32 
    # Giả lập feature ban đầu cho Node
    node_features_global = torch.ones((len(all_ips), num_node_features), dtype=torch.float32) 
    
    for current_bin in tqdm(range(max_bin + 1), desc="Quét Snapshot"):
        if current_bin in grouped.groups:
            bin_df = grouped.get_group(current_bin)
            
            # Trích xuất nhãn phổ biến nhất (mode) của các flow hợp thành Cạnh trong 1 giây
            edge_agg = bin_df.groupby(['src_ip', 'dst_ip']).agg(
                f_ij=('timestamp_num', 'count'),
                d_ij=(size_col, 'sum'),
                edge_label=(label_col, lambda x: x.mode()[0]) # Lấy nhãn đại diện cho Cạnh / Flow
            ).reset_index()
            
            for _, row in edge_agg.iterrows():
                u, v = ip_to_id[row['src_ip']], ip_to_id[row['dst_ip']]
                w_ij = alpha * row['f_ij'] + beta * row['d_ij']
                active_edges[(u, v)] = {
                    'last_bin': current_bin, 
                    'w_ij': w_ij,
                    'label': row['edge_label']
                }
                
        # Xóa cạnh hết hạn
        keys_to_remove = [k for k, info in active_edges.items() if current_bin - info['last_bin'] >= n_inactive]
        for k in keys_to_remove: del active_edges[k]
            
        if len(active_edges) > 0:
            srcs = [k[0] for k in active_edges.keys()]
            dsts = [k[1] for k in active_edges.keys()]
            weights = [info['w_ij'] for info in active_edges.values()]
            labels = [info['label'] for info in active_edges.values()]
            
            edge_index = torch.tensor([srcs, dsts], dtype=torch.long)
            edge_weight = torch.tensor(weights, dtype=torch.float32)
            
            # CHÚ Ý: y bây giờ chứa EDGE LABELS (Kích thước bằng số lượng cạnh)
            edge_labels_tensor = torch.tensor(labels, dtype=torch.long)
            
            graph = Data(x=node_features_global, edge_index=edge_index, edge_weight=edge_weight, y=edge_labels_tensor)
            snapshots.append(graph)
            
    print(f"Đã tạo xong chuỗi thời gian vòng đời gồm {len(snapshots)} Mini-Graphs!")
    return snapshots

# =================================================================================
# 3. KIẾN TRÚC MÔ HÌNH (GCN DYNAMIC - EDGE PREDICTOR)
# =================================================================================
class DynamicIoT_EdgeIDS(nn.Module):
    def __init__(self, num_node_features, num_classes):
        super(DynamicIoT_EdgeIDS, self).__init__()
        # 1. Feature -> Nhúng Node
        self.linear_in = nn.Linear(num_node_features, 32)
        
        # 2. Các tầng GCN
        self.gcn1 = GCNConv(32, 64)
        self.gcn2 = GCNConv(64, 128)
        self.gcn3 = GCNConv(128, 64)
        self.dropout = nn.Dropout(p=0.5)
        
        # 3. Phân loại Cạnh (Edge Classifier): 
        # Nối vector (Node src) + (Node dst) + (Trọng số cạnh vô hướng) = 64 + 64 + 1
        self.classifier = nn.Linear(64 + 64 + 1, num_classes)

    def encode_nodes(self, x, edge_index, edge_weight):
        """Hàm bóp nội dung Đồ thị thành các Vector Node Embedding (Tính 1 lần/Snapshot)"""
        x = self.linear_in(x)
        x = F.relu(self.gcn1(x, edge_index, edge_weight))
        x = self.dropout(x)
        x = F.relu(self.gcn2(x, edge_index, edge_weight))
        x = self.dropout(x)
        x = F.relu(self.gcn3(x, edge_index, edge_weight))
        x = self.dropout(x)
        return x

    def predict_edges(self, node_emb, edge_index, edge_weight):
        """Hàm dự đoán nhãn cho cạnh dựa trên node embedding và thuộc tính cạnh"""
        src_nodes = edge_index[0]
        dst_nodes = edge_index[1]
        
        # Nối (Concatenate) nhúng của IP gửi, IP nhận, và Trọng số Flow
        edge_repr = torch.cat([
            node_emb[src_nodes], 
            node_emb[dst_nodes], 
            edge_weight.unsqueeze(-1)
        ], dim=-1)
        
        return self.classifier(edge_repr)

# =================================================================================
# 4. QUÁ TRÌNH HUẤN LUYỆN DÀNH CHO EDGE (EDGE CLASSIFICATION TRAINING LOOP)
# =================================================================================
def train_dynamic_edge_gcn(model, train_snapshots, valid_snapshots, device, num_epochs=200, batch_size=10000):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4) 
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)
    
    best_val_loss = float('inf')
    early_stop_count = 0
    patience = 20
    
    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0.0
        all_train_preds, all_train_labels = [], []
        total_train_edges = sum(g.num_edges for g in train_snapshots)
        
        for g in train_snapshots:
            if g.num_edges == 0: continue
            g = g.to(device)
            
            optimizer.zero_grad()
            
            # 1. Lan truyền thông điệp để tạo Nhúng Nodes
            node_emb = model.encode_nodes(g.x, g.edge_index, g.edge_weight)
            
            # 2. Xử lý Mini-batch qua các Cạnh (Edge Batching để tránh OOM)
            perm = torch.randperm(g.num_edges)
            for i in range(0, g.num_edges, batch_size):
                idx = perm[i:i+batch_size]
                e_idx_batch = g.edge_index[:, idx]
                e_w_batch = g.edge_weight[idx]
                y_batch = g.y[idx] # y lúc này là Edge Labels
                
                logits = model.predict_edges(node_emb, e_idx_batch, e_w_batch)
                
                # Chiết khấu Loss theo scale mini-batch / full-batch
                loss = criterion(logits, y_batch) * (len(idx) / g.num_edges)
                loss.backward(retain_graph=True) 
                
                total_train_loss += loss.item() * g.num_edges
                preds = logits.argmax(dim=1)
                all_train_preds.extend(preds.cpu().tolist())
                all_train_labels.extend(y_batch.cpu().tolist())
                
            optimizer.step()

        train_acc = accuracy_score(all_train_labels, all_train_preds)
        train_f1 = f1_score(all_train_labels, all_train_preds, average='macro') if len(set(all_train_labels)) > 1 else 0.0
        avg_train_loss = total_train_loss / max(1, total_train_edges)
        
        # --- VALIDATION ---
        model.eval()
        total_val_loss = 0.0
        all_val_preds, all_val_labels = [], []
        total_val_edges = sum(g.num_edges for g in valid_snapshots)
        
        with torch.no_grad():
            for g in valid_snapshots:
                if g.num_edges == 0: continue
                g = g.to(device)
                
                node_emb = model.encode_nodes(g.x, g.edge_index, g.edge_weight)
                
                # Đánh giá theo block để tiết kiệm VRAM
                for i in range(0, g.num_edges, batch_size):
                    idx = torch.arange(i, min(i+batch_size, g.num_edges), device=device)
                    e_idx_batch = g.edge_index[:, idx]
                    e_w_batch = g.edge_weight[idx]
                    y_batch = g.y[idx]
                    
                    logits = model.predict_edges(node_emb, e_idx_batch, e_w_batch)
                    loss = criterion(logits, y_batch)
                    
                    total_val_loss += loss.item() * len(idx)
                    preds = logits.argmax(dim=1)
                    all_val_preds.extend(preds.cpu().tolist())
                    all_val_labels.extend(y_batch.cpu().tolist())
                    
        val_acc = accuracy_score(all_val_labels, all_val_preds)
        val_f1 = f1_score(all_val_labels, all_val_preds, average='macro') if len(set(all_val_labels)) > 1 else 0.0
        avg_val_loss = total_val_loss / max(1, total_val_edges)
        
        scheduler.step(avg_val_loss)
        
        print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f} (F1: {train_f1:.4f}) | Val Loss: {avg_val_loss:.4f} (F1: {val_f1:.4f})")
              
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            early_stop_count = 0
            torch.save(model.state_dict(), "SOTA_DynamicIDS_Edge_Best.pth")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                print(f"=== Đã kích hoạt Early Stopping tại Epoch {epoch+1}! ===")
                break

In [11]:
import gc
import torch

# Kích thước khung thời gian (delta_t) và số chu kỳ giữ kết nối (n_inactive)
DELTA_T = 1.0
N_INACTIVE = 5

# Dùng kích thước trung bình của phân đoạn tính bằng Byte làm trọng số đo lường khối lượng (Volume)
SIZE_COL = 'segment_size_mean' 
LABEL_COL = 'label'

print("--- 1. TẠO ĐỒ THỊ TẬP TRAIN ---")
train_snapshots = build_dynamic_snapshots(train_df, delta_t=DELTA_T, n_inactive=N_INACTIVE, size_col=SIZE_COL, label_col=LABEL_COL)

print("\n--- 2. TẠO ĐỒ THỊ TẬP VALID ---")
valid_snapshots = build_dynamic_snapshots(valid_df, delta_t=DELTA_T, n_inactive=N_INACTIVE, size_col=SIZE_COL, label_col=LABEL_COL)

print("\n--- 3. TẠO ĐỒ THỊ TẬP TEST ---")
test_snapshots = build_dynamic_snapshots(test_df, delta_t=DELTA_T, n_inactive=N_INACTIVE, size_col=SIZE_COL, label_col=LABEL_COL)

# Giải phóng RAM để hệ thống hoạt động mượt mà
gc.collect()

# =================================================================================
# KIỂM TRA THUỘC TÍNH CỦA ĐỒ THỊ (GRAPH DIAGNOSTICS)
# =================================================================================
print("\n=======================================================")
print("KIỂM TRA THUỘC TÍNH CỦA CÁC ĐỒ THỊ VỪA TẠO")
print("=======================================================")

def inspect_snapshots(name, snapshots):
    num_graphs = len(snapshots)
    total_edges = sum(g.num_edges for g in snapshots)
    
    if num_graphs == 0:
        print(f"- {name}: 0 đồ thị (Snapshot)!")
        return
        
    # Tìm đồ thị lớn nhất (Có nhiều cạnh nhất) để phân tích cấu trúc
    max_edge_graph = max(snapshots, key=lambda g: g.num_edges)
    
    print(f"\n[{name}] Tổng số Snapshots (Giây): {num_graphs:,} | Tổng số Edges (Flows): {total_edges:,}")
    print(f" -> Snapshot LỚN NHẤT có: {max_edge_graph.num_edges:,} cạnh (flows), {max_edge_graph.num_nodes:,} nodes (IPs)")
    print(f" -> Kích thước Cấu trúc (edge_index shape): {max_edge_graph.edge_index.shape}")
    print(f" -> Kích thước Trọng số (edge_weight shape): {max_edge_graph.edge_weight.shape}")
    print(f" -> Kích thước Nhãn (edge_labels shape)    : {max_edge_graph.y.shape}")
    
    # Kiểm tra phân phối Nhãn trên TOÀN BỘ các Snapshots
    all_labels = torch.cat([g.y for g in snapshots])
    classes, counts = torch.unique(all_labels, return_counts=True)
    
    print(f" -> Phân phối nhãn (Label Distribution) trong {name}:")
    for c, count in zip(classes.tolist(), counts.tolist()):
        pct = count / len(all_labels) * 100
        print(f"    - Lớp {c}: {count:>8,} edges ({pct:>6.2f}%)")

inspect_snapshots("TRAIN", train_snapshots)
inspect_snapshots("VALID", valid_snapshots)
inspect_snapshots("TEST", test_snapshots)

--- 1. TẠO ĐỒ THỊ TẬP TRAIN ---
Khởi tạo Dynamic Snapshots (Edge Labels) với Δt=1.0s...


Quét Snapshot:   0%|          | 0/865597 [00:00<?, ?it/s]

Đã tạo xong chuỗi thời gian vòng đời gồm 5259 Mini-Graphs!

--- 2. TẠO ĐỒ THỊ TẬP VALID ---
Khởi tạo Dynamic Snapshots (Edge Labels) với Δt=1.0s...


Quét Snapshot:   0%|          | 0/865722 [00:00<?, ?it/s]

Đã tạo xong chuỗi thời gian vòng đời gồm 1157 Mini-Graphs!

--- 3. TẠO ĐỒ THỊ TẬP TEST ---
Khởi tạo Dynamic Snapshots (Edge Labels) với Δt=1.0s...


Quét Snapshot:   0%|          | 0/865932 [00:00<?, ?it/s]

Đã tạo xong chuỗi thời gian vòng đời gồm 1566 Mini-Graphs!

KIỂM TRA THUỘC TÍNH CỦA CÁC ĐỒ THỊ VỪA TẠO

[TRAIN] Tổng số Snapshots (Giây): 5,259 | Tổng số Edges (Flows): 49,209
 -> Snapshot LỚN NHẤT có: 21 cạnh (flows), 23 nodes (IPs)
 -> Kích thước Cấu trúc (edge_index shape): torch.Size([2, 21])
 -> Kích thước Trọng số (edge_weight shape): torch.Size([21])
 -> Kích thước Nhãn (edge_labels shape)    : torch.Size([21])
 -> Phân phối nhãn (Label Distribution) trong TRAIN:
    - Lớp 0:      566 edges (  1.15%)
    - Lớp 1:   15,792 edges ( 32.09%)
    - Lớp 2:    1,379 edges (  2.80%)
    - Lớp 3:   10,295 edges ( 20.92%)
    - Lớp 4:      297 edges (  0.60%)
    - Lớp 5:      529 edges (  1.08%)
    - Lớp 6:      616 edges (  1.25%)
    - Lớp 7:   17,228 edges ( 35.01%)
    - Lớp 8:      814 edges (  1.65%)
    - Lớp 9:    1,274 edges (  2.59%)
    - Lớp 10:      419 edges (  0.85%)

[VALID] Tổng số Snapshots (Giây): 1,157 | Tổng số Edges (Flows): 10,723
 -> Snapshot LỚN NHẤT có: 17 cạnh

In [ ]:
import numpy as np
from sklearn.utils.class_weight import compute_class_weight
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch_geometric.nn import GCNConv
from sklearn.metrics import accuracy_score, f1_score, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import torch

# =================================================================================
# BƯỚC 1: XÁC ĐỊNH THIẾT BỊ TÍNH TOÁN VÀ SỐ LỚP
# =================================================================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"-> Đang sử dụng thiết bị: {device}")

all_train_labels = torch.cat([g.y.cpu() for g in train_snapshots]).numpy()
unique_classes = np.unique(all_train_labels)
num_classes = len(unique_classes)
num_node_features = 32
print(f"-> Tổng số nhãn (classes): {num_classes}")

# =================================================================================
# BƯỚC 1.5: SIÊU CHUẨN HÓA TRỌNG SỐ
# =================================================================================
print("-> Đang làm sạch và siêu chuẩn hóa dữ liệu đồ thị...")
for snapshots in [train_snapshots, valid_snapshots, test_snapshots]:
    for g in snapshots:
        if g.num_edges > 0:
            # Sửa tận gốc: Mọi giá trị hỏng của edge_weight chuyển về 0.0
            g.edge_weight = torch.nan_to_num(g.edge_weight, nan=0.0, posinf=0.0, neginf=0.0)
            # Log1p an toàn để kéo số nhỏ lại
            g.edge_weight = torch.log1p(torch.clamp(g.edge_weight, min=0.0))
            
# BẢO VỆ CLASS WEIGHTS
class_weights = compute_class_weight(class_weight='balanced', classes=unique_classes, y=all_train_labels)
# Chặn đứng các tình huống Class Weights trả về NaN hoặc quá lớn (gây nổ Loss)
class_weights = np.nan_to_num(class_weights, nan=1.0, posinf=10.0, neginf=1.0)
class_weights = np.clip(class_weights, a_min=0.1, a_max=20.0) 
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)

# =================================================================================
# BƯỚC 3: KIẾN TRÚC MÔ HÌNH CHỐNG NAN TUYỆT ĐỐI (2 GCN LAYERS)
# =================================================================================
class Robust_DynamicIoT_EdgeIDS(nn.Module):
    def __init__(self, num_node_features, num_classes):
        super().__init__()
        self.linear_in = nn.Linear(num_node_features, 32)
        
        # Chỉ dùng 2 lớp GCN để tránh Oversmoothing (xóa nhòa đặc trưng giữa Hacker và Victim)
        self.gcn1 = GCNConv(32, 64)
        self.norm1 = nn.LayerNorm(64)
        
        self.gcn2 = GCNConv(64, 64)
        self.norm2 = nn.LayerNorm(64)
        
        self.dropout = nn.Dropout(p=0.4)
        self.classifier = nn.Linear(64 + 64 + 1, num_classes)

    def encode_nodes(self, x, edge_index):
        x = self.linear_in(x)
        
        # Layer 1
        x = self.gcn1(x, edge_index)
        x = self.norm1(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        # Layer 2
        x = self.gcn2(x, edge_index)
        x = self.norm2(x)
        x = F.relu(x)
        x = self.dropout(x)
        
        return x

    def predict_edges(self, node_emb, edge_index, edge_weight):
        src_nodes = edge_index[0]
        dst_nodes = edge_index[1]
        
        # Trọng số Cạnh chỉ được bổ sung vào đây (Safe space)
        edge_repr = torch.cat([
            node_emb[src_nodes], 
            node_emb[dst_nodes], 
            edge_weight.unsqueeze(-1)
        ], dim=-1)
        
        return self.classifier(edge_repr)

# =================================================================================
# BƯỚC 4: HÀM HUẤN LUYỆN DO THỜI GIAN THỰC
# =================================================================================
def train_dynamic_edge_gcn_weighted(model, train_snapshots, valid_snapshots, device, class_weights, num_epochs=100):
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4) 
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=5, verbose=True)
    
    best_val_loss = float('inf')
    early_stop_count = 0
    patience = 15
    
    for epoch in range(num_epochs):
        model.train()
        total_train_loss = 0.0
        all_train_preds, all_train_labels = [], []
        total_train_edges = sum(g.num_edges for g in train_snapshots)
        
        for step, g in enumerate(train_snapshots):
            if g.num_edges == 0: continue
            g = g.to(device)
            
            optimizer.zero_grad(set_to_none=True)
            
            node_emb = model.encode_nodes(g.x, g.edge_index) # Topology Only
            logits = model.predict_edges(node_emb, g.edge_index, g.edge_weight) # Edge logic
            
            loss = criterion(logits, g.y)
            loss.backward()
            
            # Clip gradient trực tiếp trên từng đồ thị để tránh Gradient Explosion
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            total_train_loss += (loss.item() * g.num_edges)
            preds = logits.argmax(dim=1)
            all_train_preds.extend(preds.cpu().tolist())
            all_train_labels.extend(g.y.cpu().tolist())

        train_f1 = f1_score(all_train_labels, all_train_preds, average='macro', zero_division=0)
        avg_train_loss = total_train_loss / max(1, total_train_edges)
        
        # --- VALIDATION ---
        model.eval()
        total_val_loss = 0.0
        all_val_preds, all_val_labels = [], []
        total_val_edges = sum(g.num_edges for g in valid_snapshots)
        
        with torch.no_grad():
            for g in valid_snapshots:
                if g.num_edges == 0: continue
                g = g.to(device)
                
                node_emb = model.encode_nodes(g.x, g.edge_index)
                logits = model.predict_edges(node_emb, g.edge_index, g.edge_weight)
                loss = criterion(logits, g.y)
                
                total_val_loss += loss.item() * g.num_edges
                preds = logits.argmax(dim=1)
                all_val_preds.extend(preds.cpu().tolist())
                all_val_labels.extend(g.y.cpu().tolist())
                    
        val_f1 = f1_score(all_val_labels, all_val_preds, average='macro', zero_division=0)
        avg_val_loss = total_val_loss / max(1, total_val_edges)
        
        scheduler.step(avg_val_loss)
        print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f} (F1 Macro: {train_f1:.4f}) | Val Loss: {avg_val_loss:.4f} (F1 Macro: {val_f1:.4f})")
              
        if avg_val_loss < best_val_loss and not np.isnan(avg_val_loss):
            best_val_loss = avg_val_loss
            early_stop_count = 0
            torch.save(model.state_dict(), "SOTA_DynamicIDS_Edge_Best.pth")
        else:
            early_stop_count += 1
            if early_stop_count >= patience:
                print(f"=== Đã kích hoạt Early Stopping tại Epoch {epoch+1}! ===")
                break

# =================================================================================
# BẮT ĐẦU HUẤN LUYỆN
# =================================================================================
model = Robust_DynamicIoT_EdgeIDS(num_node_features=num_node_features, num_classes=num_classes).to(device)

print("\n--- BẮT ĐẦU HUẤN LUYỆN ---")
train_dynamic_edge_gcn_weighted(
    model=model, 
    train_snapshots=train_snapshots, 
    valid_snapshots=valid_snapshots, 
    device=device, 
    class_weights=class_weights_tensor, 
    num_epochs=150
)

# =================================================================================
# KIỂM THỬ TRÊN TẬP TEST VÀ ĐÁNH GIÁ (TESTING)
# =================================================================================
print("\n--- KIỂM THỬ TRÊN TẬP TEST ---")
try:
    model.load_state_dict(torch.load("SOTA_DynamicIDS_Edge_Best.pth", map_location=device))
except:
    print("Không tải được file Weight, giữ nguyên Weight hiện tại.")
model.eval()

all_test_preds, all_test_labels = [], []
with torch.no_grad():
    for g in test_snapshots:
        if g.num_edges == 0: continue
        g = g.to(device)
        
        node_emb = model.encode_nodes(g.x, g.edge_index)
        logits = model.predict_edges(node_emb, g.edge_index, g.edge_weight)
        preds = logits.argmax(dim=1)
        
        all_test_preds.extend(preds.cpu().tolist())
        all_test_labels.extend(g.y.cpu().tolist())

target_names = [f"Lớp {i}" for i in range(num_classes)]
print("\nBÁO CÁO PHÂN LOẠI (CLASSIFICATION REPORT) TRÊN TẬP TEST:")
print(classification_report(all_test_labels, all_test_preds, target_names=target_names, zero_division=0))